In [1]:
import os
import warnings
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import geopandas as gpd
import igraph as ig
import leidenalg as la
import libpysal as lps
import esda
from tqdm import tqdm

from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import make_valid
from sklearn.metrics import normalized_mutual_info_score, fowlkes_mallows_score

warnings.filterwarnings("ignore", category=RuntimeWarning)
os.chdir('D:/urban_hierarchy_congestion')

# Mobility-flow-based hierarchy identification

In [ ]:
# =========================================================
# 1. Consensus Leiden
# =========================================================
def _run_leiden_once(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    seed: int
):
    kwargs = dict(
        weights=weights,
        n_iterations=-1,
        seed=seed
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution
    return la.find_partition(G, partition_type, **kwargs)


def _multi_run_memberships(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    n_runs: int,
    seed0: int = 0
) -> List[List[int]]:
    membs = []
    for r in range(n_runs):
        part = _run_leiden_once(G, partition_type, resolution, weights, seed0 + r)
        membs.append(list(part.membership))
    return membs


def _coassoc_from_memberships(membs: List[List[int]]) -> np.ndarray:
    n = len(membs[0])
    P = np.zeros((n, n), dtype=np.float64)

    for m in membs:
        buckets: Dict[int, list] = {}
        for idx, c in enumerate(m):
            buckets.setdefault(c, []).append(idx)
        for idxs in buckets.values():
            idxs = np.asarray(idxs, dtype=int)
            P[np.ix_(idxs, idxs)] += 1.0

    P /= float(len(membs))
    np.fill_diagonal(P, 0.0)
    P = 0.5 * (P + P.T)
    return P


def _consensus_on_coassoc(
    P: np.ndarray,
    partition_type,
    resolution: float,
    threshold: Optional[float] = None
) -> Tuple[List[int], ig.Graph]:
    P_use = P.copy()
    if threshold is not None:
        P_use[P_use < threshold] = 0.0

    Gc = ig.Graph.Weighted_Adjacency(
        P_use.tolist(),
        mode="UNDIRECTED",
        attr="weight",
        loops=False
    )

    kwargs = dict(
        weights="weight",
        n_iterations=-1,
        seed=0
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution

    part = la.find_partition(Gc, partition_type, **kwargs)
    return list(part.membership), Gc


def _similarity(m1: List[int], m2: List[int], metric: str = "NMI") -> float:
    if metric.upper() == "NMI":
        return normalized_mutual_info_score(m1, m2)
    elif metric.upper() == "FMI":
        return fowlkes_mallows_score(m1, m2)
    else:
        raise ValueError("metric must be 'NMI' or 'FMI'.")


def iterative_consensus_leiden(
    G: ig.Graph,
    partition_type=la.ModularityVertexPartition,
    resolution: float = 1.0,
    weights: Optional[str] = "weight",
    n_runs: int = 100,
    seed0: int = 0,
    max_iter: int = 10,
    tol: float = 0.01,
    metric: str = "NMI",
    threshold: Optional[float] = None,
    return_history: bool = True
):
    if G is None or G.vcount() == 0:
        return {"membership": [], "history": [], "coassoc": None}

    membs0 = _multi_run_memberships(G, partition_type, resolution, weights, n_runs, seed0)
    P = _coassoc_from_memberships(membs0)
    memb_prev, Gc = _consensus_on_coassoc(P, partition_type, resolution, threshold)
    hist = [{"iter": 0, "similarity": np.nan, "n_comms": len(set(memb_prev))}]

    for it in range(1, max_iter + 1):
        membs = _multi_run_memberships(Gc, partition_type, resolution, "weight", n_runs, seed0 + it * 1000)
        P_next = _coassoc_from_memberships(membs)
        memb_next, Gc_next = _consensus_on_coassoc(P_next, partition_type, resolution, threshold)

        sim = _similarity(memb_prev, memb_next, metric=metric)
        hist.append({"iter": it, "similarity": sim, "n_comms": len(set(memb_next))})

        if 1.0 - sim < tol:
            return {
                "membership": memb_next,
                "history": hist if return_history else None,
                "coassoc": P_next
            }

        memb_prev, Gc, P = memb_next, Gc_next, P_next

    return {
        "membership": memb_prev,
        "history": hist if return_history else None,
        "coassoc": P
    }


# =========================================================
# 2. Geometry helpers
# =========================================================
def fill_holes(geom):
    if geom is None:
        return None

    geom = make_valid(geom)

    if geom.geom_type == "Polygon":
        return Polygon(geom.exterior)

    if geom.geom_type == "MultiPolygon":
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])

    return geom


def split_region_into_spatial_components(
    unit_region: gpd.GeoDataFrame,
    region_label: str,
    fill_component_holes: bool = False
) -> gpd.GeoDataFrame:
    """
    Divide the spatial unit corresponding to a community into multiple contiguous subregions based on spatial connectivity
    """
    if len(unit_region) == 0:
        return gpd.GeoDataFrame(
            columns=["component_id", "geometry"],
            geometry="geometry",
            crs=unit_region.crs
        )

    merged = unit_region.union_all()
    comps = gpd.GeoDataFrame(geometry=[merged], crs=unit_region.crs)
    comps = comps.explode(ignore_index=True, index_parts=False)

    if fill_component_holes:
        comps["geometry"] = comps["geometry"].apply(fill_holes)

    comps["component_id"] = [f"{region_label}_c{i+1}" for i in range(len(comps))]
    return comps[["component_id", "geometry"]].copy()


def assign_units_to_spatial_components(
    unit_region: gpd.GeoDataFrame,
    components: gpd.GeoDataFrame
) -> pd.DataFrame:
    """
    Mapping a cell to a subregion of a continuous space
    """
    if len(unit_region) == 0 or len(components) == 0:
        return pd.DataFrame(columns=["id", "component_id"])

    joined = gpd.sjoin(
        unit_region[["id", "geometry"]],
        components[["component_id", "geometry"]],
        how="inner",
        predicate="intersects"
    )

    joined = joined[["id", "component_id"]].drop_duplicates().copy()
    joined.index = range(len(joined))
    return joined


# =========================================================
# 3. Moran hotspot detection
# =========================================================
def identify_inflow_core(
    od: pd.DataFrame,
    unit: gpd.GeoDataFrame,
    crit_value: float = 0.05,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
    select_mode: str = "cumulative",   # "top1" or "cumulative"
    cum_share_threshold: float = 0.8
) -> gpd.GeoDataFrame:
    """
    Within the current region:
    1. Calculate the Local Moran's I using inflow density
    2. Select the "High-High" cells
    3. Merge them into hotspot clusters
    4. Retain the main clusters according to the rules

    select_mode:
        - "top1": Keep only the cluster with the highest inflow
        - "cumulative": Sort by inflow and keep clusters until the cumulative inflow reaches the threshold
    """
    if len(unit) == 0 or len(od) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    d = od.groupby("d_id", as_index=False)["flow"].sum()
    d.rename(columns={"d_id": "id", "flow": "inflow"}, inplace=True)

    unit2 = pd.merge(unit, d, on="id", how="left")
    unit2.fillna(0, inplace=True)
    unit2["inflow_den"] = unit2["inflow"] / unit2["area"]

    try:
        if use_queen:
            w = lps.weights.Queen.from_dataframe(unit2, use_index=True, silence_warnings=True)
        else:
            w = lps.weights.Rook.from_dataframe(unit2, use_index=True, silence_warnings=True)

        y = unit2["inflow_den"]
        lm = esda.Moran_Local(
            y,
            w,
            transformation="r",
            permutations=999,
            n_jobs=-1,
            seed=42
        )
        unit2["lisa"] = lm.get_cluster_labels(crit_value=crit_value)
    except Exception:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh = unit2.loc[unit2["lisa"] == "High-High"].copy()
    if len(unit_hh) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh.index = range(len(unit_hh))

    clusters = gpd.GeoDataFrame(geometry=[unit_hh.union_all()], crs=unit.crs)
    clusters = clusters.explode(ignore_index=True, index_parts=False)

    if fill_cluster_holes:
        clusters["geometry"] = clusters["geometry"].apply(fill_holes)

    clusters["cid"] = clusters.index + 1

    hhc = gpd.overlay(unit_hh, clusters, how="intersection", keep_geom_type=True)
    hhcg = hhc.groupby("cid", as_index=False)[["inflow", "area"]].sum()

    clusters = pd.merge(clusters, hhcg, on="cid", how="inner")
    if len(clusters) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    clusters = clusters.sort_values("inflow", ascending=False, ignore_index=True).copy()

    # -------- selection rule --------
    if select_mode == "top1":
        clusters = clusters.head(1).copy()

    elif select_mode == "cumulative":
        total_inflow = clusters["inflow"].sum()
        if total_inflow <= 0:
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

        clusters["cum_share"] = clusters["inflow"].cumsum() / total_inflow
        keep_n = min((clusters["cum_share"] < cum_share_threshold).sum() + 1, len(clusters))
        clusters = clusters.head(keep_n).copy()

    else:
        raise ValueError("select_mode must be 'top1' or 'cumulative'")

    clusters.index = range(len(clusters))
    clusters.drop(columns=[c for c in ["cid", "cum_share"] if c in clusters.columns], inplace=True)

    if len(clusters) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    return clusters


# =========================================================
# 4. Graph helpers
# =========================================================
def induced_od(od: pd.DataFrame, node_ids: List[str]) -> pd.DataFrame:
    node_set = set(node_ids)
    out = od.loc[od["o_id"].isin(node_set) & od["d_id"].isin(node_set)].copy()
    out.index = range(len(out))
    return out


def build_graph_from_od(od_sub: pd.DataFrame) -> Optional[ig.Graph]:
    if len(od_sub) == 0:
        return None

    nodes = pd.unique(pd.concat([od_sub["o_id"], od_sub["d_id"]], axis=0))
    if len(nodes) < 2:
        return None

    G = ig.Graph.DataFrame(
        od_sub[["o_id", "d_id", "flow"]],
        directed=True,
        use_vids=False
    )
    G.es["weight"] = od_sub["flow"].tolist()
    return G


# =========================================================
# 5. Recursive hotspot + consensus community
# =========================================================
def recursive_hotspot_community(
    od_all: pd.DataFrame,
    unit_all: gpd.GeoDataFrame,
    node_ids: List[str],
    crit_value: float,
    cum_share_threshold: float,
    use_queen: bool,
    fill_cluster_holes: bool,
    n_runs: int,
    tol: float,
    resolution: float,
    min_units: int,
    level: int,
    path_prefix: str,
    unit_labels: pd.DataFrame,
    region_polygons: Dict[int, List[gpd.GeoDataFrame]],
    diagnostics: List[dict],
):
    """
    Logic: 
    1. Stop when the number of cells in the current region is insufficient; do not perform a hotspot split.
    2. The hotspot split is the primary step for identifying centers.
    3. After removing the center, if the number of remaining cells is insufficient, stop and do not perform a community split.
    4. Consensus communities only determine whether to partition.
       - n_comms >= 2: First partition by functional communities, then split by spatial connectivity, and continue recursively.
       - n_comms < 2: Do not partition; continue recursion on the whole.
    """

    unit_sub = unit_all.loc[unit_all["id"].isin(node_ids)].copy()
    unit_sub.index = range(len(unit_sub))
    od_sub = induced_od(od_all, node_ids)

    if len(unit_sub) == 0:
        return

    # -------------------------------------------------
    # Step 0. stop if too few units for Moran hotspot detection
    # -------------------------------------------------
    if len(unit_sub) < min_units:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1" if level == 1 else "cumulative",
            "reason": "too_few_units_for_moran"
        })
        return

    # -------------------------------------------------
    # Step 1. identify hotspot centers
    # -------------------------------------------------
    if level == 1:
        centers = identify_inflow_core(
            od_sub,
            unit_sub,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            select_mode="top1"
        )
        selection_mode = "top1"
    else:
        centers = identify_inflow_core(
            od_sub,
            unit_sub,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            select_mode="cumulative",
            cum_share_threshold=cum_share_threshold
        )
        selection_mode = "cumulative"

    if len(centers) == 0:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "no_hotspot"
        })
        return

    unit_center = gpd.overlay(
        unit_sub[["id", "geometry"]],
        centers[["geometry"]],
        how="intersection",
        keep_geom_type=True
    )
    center_ids = unit_center["id"].astype(str).unique().tolist()

    # Write to level only for cells that have not yet been assigned a value
    mask = unit_labels["id"].isin(center_ids)
    unit_labels.loc[mask & unit_labels["level"].isna(), "level"] = level

    # -------------------------------------------------
    # Step 2. remove centers
    # -------------------------------------------------
    rem_ids = [x for x in node_ids if x not in set(center_ids)]
    if len(rem_ids) == 0:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": 0,
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "all_center"
        })
        return

    # -------------------------------------------------
    # Step 2.5. stop if too few residual units for split
    # -------------------------------------------------
    if len(rem_ids) < min_units:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "too_few_units_for_split"
        })
        return

    od_rem = induced_od(od_all, rem_ids)
    G = build_graph_from_od(od_rem)

    if G is None:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "graph_none"
        })
        return

    # -------------------------------------------------
    # Step 3. consensus community on residual network
    # -------------------------------------------------
    res = iterative_consensus_leiden(
        G,
        partition_type=la.ModularityVertexPartition,
        resolution=resolution,
        weights="weight",
        n_runs=n_runs,
        tol=tol,
        metric="NMI",
        max_iter=10
    )

    membership = res["membership"]
    n_comms = len(set(membership))

    # -------------------------------------------------
    # Step 4. only one community: no partition, continue globally
    # -------------------------------------------------
    if n_comms < 2:
        diagnostics.append({
            "path": path_prefix if path_prefix != "" else "ROOT",
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": n_comms,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": selection_mode,
            "reason": "single_comm"
        })

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=rem_ids,
            crit_value=crit_value,
            cum_share_threshold=cum_share_threshold,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            min_units=min_units,
            level=level + 1,
            path_prefix=path_prefix,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )
        return

    # -------------------------------------------------
    # Step 5. multiple communities:
    # first split each community into spatially connected components,
    # then recurse on each connected component
    # -------------------------------------------------
    name_to_comm = dict(zip(G.vs["name"], membership))

    rem_df = pd.DataFrame({
        "id": list(name_to_comm.keys()),
        "comm_id": [name_to_comm[x] for x in name_to_comm.keys()]
    })

    if path_prefix == "":
        rem_df["comm_label"] = rem_df["comm_id"].astype(int).astype(str)
    else:
        rem_df["comm_label"] = path_prefix + "_" + rem_df["comm_id"].astype(int).astype(str)

    region_col = f"region_{level}"
    if region_col not in unit_labels.columns:
        unit_labels[region_col] = np.nan

    component_records = []
    polygon_records = []
    n_components_total = 0

    for comm_label in rem_df["comm_label"].dropna().unique().tolist():
        comm_ids = rem_df.loc[rem_df["comm_label"] == comm_label, "id"].astype(str).tolist()

        unit_comm = unit_all.loc[unit_all["id"].isin(comm_ids), ["id", "geometry"]].copy()
        unit_comm.index = range(len(unit_comm))

        comps = split_region_into_spatial_components(
            unit_comm,
            region_label=comm_label,
            fill_component_holes=False
        )

        if len(comps) == 0:
            continue

        n_components_total += len(comps)

        assign_df = assign_units_to_spatial_components(unit_comm, comps)
        if len(assign_df) == 0:
            continue

        component_records.append(assign_df)

        comp_poly = comps.copy()
        comp_poly.rename(columns={"component_id": "region_id"}, inplace=True)
        comp_poly["level"] = level
        polygon_records.append(comp_poly)

    diagnostics.append({
        "path": path_prefix if path_prefix != "" else "ROOT",
        "level": level,
        "n_units": len(node_ids),
        "n_centers": len(center_ids),
        "n_remaining": len(rem_ids),
        "n_comms": n_comms,
        "n_spatial_components": n_components_total,
        "continue_split": n_components_total >= 1,
        "selection_mode": selection_mode,
        "reason": "split"
    })

    if len(component_records) == 0:
        return

    rem_component_df = pd.concat(component_records, axis=0, ignore_index=True)
    rem_component_df = rem_component_df.rename(columns={"component_id": region_col})

    # Write the `region_level` tag in place
    mapping = dict(zip(rem_component_df["id"], rem_component_df[region_col]))
    unit_labels[region_col] = unit_labels["id"].map(mapping).combine_first(unit_labels[region_col])

    # Save the first two layers of polygons
    if level <= 2 and len(polygon_records) > 0:
        poly = pd.concat(polygon_records, axis=0, ignore_index=True)
        poly = gpd.GeoDataFrame(poly, geometry="geometry", crs=unit_all.crs)
        region_polygons.setdefault(level, []).append(poly)

    # Recursively enter each consecutive subregion
    child_regions = rem_component_df[region_col].dropna().unique().tolist()

    for rg in child_regions:
        child_ids = rem_component_df.loc[
            rem_component_df[region_col] == rg, "id"
        ].astype(str).tolist()

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=child_ids,
            crit_value=crit_value,
            cum_share_threshold=cum_share_threshold,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            min_units=min_units,
            level=level + 1,
            path_prefix=rg,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )


# =========================================================
# 6. Main pipeline
# =========================================================
def detect_hotspot_community_hierarchy(
    unit: gpd.GeoDataFrame,
    od: pd.DataFrame,
    crit_value: float = 0.05,
    cum_share_threshold: float = 0.8,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
    n_runs: int = 100,
    tol: float = 0.01,
    resolution: float = 1.0,
    min_units: int = 10
):
    """
    Returns: 
    - unit_result: The level, region_1, region_2, ... for each unit 
    - region_polygons_all: Polygons resulting from dissolving the first two layers 
    - diagnostics_df
    """

    unit = unit.copy()
    od = od.copy()

    unit["id"] = unit["id"].astype(str)
    od["o_id"] = od["o_id"].astype(str)
    od["d_id"] = od["d_id"].astype(str)

    if "area" not in unit.columns:
        unit["area"] = unit.geometry.area / 1e6

    keep_ids = pd.unique(pd.concat([od["o_id"], od["d_id"]], axis=0))
    unit = unit.loc[unit["id"].isin(keep_ids)].copy()
    unit.index = range(len(unit))

    unit_labels = unit[["id"]].copy()
    unit_labels["level"] = np.nan

    region_polygons = {}
    diagnostics = []

    root_ids = sorted(unit["id"].unique().tolist())

    recursive_hotspot_community(
        od_all=od,
        unit_all=unit,
        node_ids=root_ids,
        crit_value=crit_value,
        cum_share_threshold=cum_share_threshold,
        use_queen=use_queen,
        fill_cluster_holes=fill_cluster_holes,
        n_runs=n_runs,
        tol=tol,
        resolution=resolution,
        min_units=min_units,
        level=1,
        path_prefix="",
        unit_labels=unit_labels,
        region_polygons=region_polygons,
        diagnostics=diagnostics,
    )

    unit_result = pd.merge(unit, unit_labels, on="id", how="left")

    max_center_level = int(unit_result["level"].dropna().max()) if unit_result["level"].notna().any() else 0
    unit_result.loc[unit_result["level"].isna(), "level"] = max_center_level + 1
    unit_result["level"] = unit_result["level"].astype(int)

    region_polygons_all = {}
    for lv, polys in region_polygons.items():
        if len(polys) == 0:
            continue
        g = pd.concat(polys, axis=0, ignore_index=True)
        g = gpd.GeoDataFrame(g, geometry="geometry", crs=unit.crs)
        region_polygons_all[lv] = g

    diagnostics_df = pd.DataFrame(diagnostics)

    return unit_result, region_polygons_all, diagnostics_df


# =========================================================
# 7. Save outputs
# =========================================================
def save_hotspot_community_outputs(
    unit_result: gpd.GeoDataFrame,
    region_polygons_all: Dict[int, gpd.GeoDataFrame],
    diagnostics_df: pd.DataFrame,
    out_dir: str,
    prefix: str
):
    os.makedirs(out_dir, exist_ok=True)

    unit_result.to_file(os.path.join(out_dir, f"{prefix}_unit_hierarchy.shp"))

    if 1 in region_polygons_all:
        region_polygons_all[1].to_file(os.path.join(out_dir, f"{prefix}_region_level_1.shp"))

    if 2 in region_polygons_all:
        region_polygons_all[2].to_file(os.path.join(out_dir, f"{prefix}_region_level_2.shp"))

    diagnostics_df.to_csv(os.path.join(out_dir, f"{prefix}_diagnostics.csv"), index=False)


# =========================================================
# 8. Example usage
# =========================================================
def get_unit_name(city: str) -> str:
    if city in ["beijing", "shanghai", "shenzhen", "nanjing"]:
        return "grid_1k"
    elif city == "london":
        return "msoa"
    elif city in ["losangeles", "newyork"]:
        return "tract"
    else:
        raise ValueError(f"Unknown city: {city}")

In [ ]:
if __name__ == "__main__":
    for city in ["beijing","shanghai","shenzhen"]:
        unit_name = get_unit_name(city)

        od = pd.read_csv(f"D:/urban_hierarchy_congestion/data/od_tables/{city}_od.csv")
        od = od.loc[(od["o_id"] != od["d_id"]) & (od["flow"] > 0)].copy()
        od.index = range(len(od))

        unit = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{unit_name}.shp")
        unit["area"] = unit.geometry.area / 1e6
        unit = unit[["id", "area", "geometry"]].copy()
        unit = unit.loc[unit['id'].isin(od['o_id']) | unit['id'].isin(od['d_id'])].copy()
        unit.index = range(len(unit))

        unit_result, region_polygons_all, diagnostics_df = detect_hotspot_community_hierarchy(
            unit=unit,
            od=od,
            crit_value=0.05,
            cum_share_threshold=0.8,
            use_queen=False,
            fill_cluster_holes=True,
            n_runs=100,
            tol=0.01,
            resolution=1.0,
            min_units=10
        )

        save_hotspot_community_outputs(
            unit_result=unit_result,
            region_polygons_all=region_polygons_all,
            diagnostics_df=diagnostics_df,
            out_dir="D:/urban_hierarchy_congestion/results/hierarchy_identification_results",
            prefix=city
        )
    
    for city in ["london","losangeles","newyork"]:
        unit_name = get_unit_name(city)

        od = pd.read_csv(f"D:/urban_hierarchy_congestion/data/od_tables/{city}_od.csv")
        od = od.loc[(od["o_id"] != od["d_id"]) & (od["flow"] > 0)].copy()
        od.index = range(len(od))

        unit = gpd.read_file(f"D:/urban_hierarchy_congestion/taz/{city}_{unit_name}.shp")
        unit["area"] = unit.geometry.area / 1e6
        unit = unit[["id", "area", "geometry"]].copy()
        unit = unit.loc[unit['id'].isin(od['o_id']) | unit['id'].isin(od['d_id'])].copy()
        unit.index = range(len(unit))

        unit_result, region_polygons_all, diagnostics_df = detect_hotspot_community_hierarchy(
            unit=unit,
            od=od,
            crit_value=0.05,
            cum_share_threshold=0.8,
            use_queen=True,
            fill_cluster_holes=True,
            n_runs=100,
            tol=0.01,
            resolution=1.0,
            min_units=10
        )

        save_hotspot_community_outputs(
            unit_result=unit_result,
            region_polygons_all=region_polygons_all,
            diagnostics_df=diagnostics_df,
            out_dir="D:/urban_hierarchy_congestion/results/hierarchy_identification_results",
            prefix=city
        )

In [ ]:
import os
import warnings
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import geopandas as gpd
import igraph as ig
import leidenalg as la
import libpysal as lps
import esda

from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import make_valid
from sklearn.metrics import normalized_mutual_info_score, fowlkes_mallows_score

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# 1. Consensus Leiden
# =========================================================
def _run_leiden_once(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    seed: int
):
    kwargs = dict(
        weights=weights,
        n_iterations=-1,
        seed=seed
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution
    return la.find_partition(G, partition_type, **kwargs)


def _multi_run_memberships(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    n_runs: int,
    seed0: int = 0
) -> List[List[int]]:
    membs = []
    for r in range(n_runs):
        part = _run_leiden_once(G, partition_type, resolution, weights, seed0 + r)
        membs.append(list(part.membership))
    return membs


def _coassoc_from_memberships(membs: List[List[int]]) -> np.ndarray:
    n = len(membs[0])
    P = np.zeros((n, n), dtype=np.float64)

    for m in membs:
        buckets: Dict[int, list] = {}
        for idx, c in enumerate(m):
            buckets.setdefault(c, []).append(idx)
        for idxs in buckets.values():
            idxs = np.asarray(idxs, dtype=int)
            P[np.ix_(idxs, idxs)] += 1.0

    P /= float(len(membs))
    np.fill_diagonal(P, 0.0)
    P = 0.5 * (P + P.T)
    return P


def _consensus_on_coassoc(
    P: np.ndarray,
    partition_type,
    resolution: float,
    threshold: Optional[float] = None
) -> Tuple[List[int], ig.Graph]:
    P_use = P.copy()
    if threshold is not None:
        P_use[P_use < threshold] = 0.0

    Gc = ig.Graph.Weighted_Adjacency(
        P_use.tolist(),
        mode="UNDIRECTED",
        attr="weight",
        loops=False
    )

    kwargs = dict(
        weights="weight",
        n_iterations=-1,
        seed=0
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution

    part = la.find_partition(Gc, partition_type, **kwargs)
    return list(part.membership), Gc


def _similarity(m1: List[int], m2: List[int], metric: str = "NMI") -> float:
    if metric.upper() == "NMI":
        return normalized_mutual_info_score(m1, m2)
    elif metric.upper() == "FMI":
        return fowlkes_mallows_score(m1, m2)
    else:
        raise ValueError("metric must be 'NMI' or 'FMI'.")


def iterative_consensus_leiden(
    G: ig.Graph,
    partition_type=la.ModularityVertexPartition,
    resolution: float = 1.0,
    weights: Optional[str] = "weight",
    n_runs: int = 100,
    seed0: int = 0,
    max_iter: int = 10,
    tol: float = 0.01,
    metric: str = "NMI",
    threshold: Optional[float] = None,
    return_history: bool = True
):
    if G is None or G.vcount() == 0:
        return {"membership": [], "history": [], "coassoc": None}

    membs0 = _multi_run_memberships(G, partition_type, resolution, weights, n_runs, seed0)
    P = _coassoc_from_memberships(membs0)
    memb_prev, Gc = _consensus_on_coassoc(P, partition_type, resolution, threshold)
    hist = [{"iter": 0, "similarity": np.nan, "n_comms": len(set(memb_prev))}]

    for it in range(1, max_iter + 1):
        membs = _multi_run_memberships(Gc, partition_type, resolution, "weight", n_runs, seed0 + it * 1000)
        P_next = _coassoc_from_memberships(membs)
        memb_next, Gc_next = _consensus_on_coassoc(P_next, partition_type, resolution, threshold)

        sim = _similarity(memb_prev, memb_next, metric=metric)
        hist.append({"iter": it, "similarity": sim, "n_comms": len(set(memb_next))})

        if 1.0 - sim < tol:
            return {
                "membership": memb_next,
                "history": hist if return_history else None,
                "coassoc": P_next
            }

        memb_prev, Gc, P = memb_next, Gc_next, P_next

    return {
        "membership": memb_prev,
        "history": hist if return_history else None,
        "coassoc": P
    }


# =========================================================
# 2. Geometry helpers
# =========================================================
def fill_holes(geom):
    if geom is None:
        return None

    geom = make_valid(geom)

    if geom.geom_type == "Polygon":
        return Polygon(geom.exterior)

    if geom.geom_type == "MultiPolygon":
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])

    return geom


def split_region_into_spatial_components(
    unit_region: gpd.GeoDataFrame,
    region_label: str,
    fill_component_holes: bool = False
) -> gpd.GeoDataFrame:
    """
    将一个 community 对应的空间单元按空间连通性拆成多个连续子区
    """
    if len(unit_region) == 0:
        return gpd.GeoDataFrame(
            columns=["component_id", "geometry"],
            geometry="geometry",
            crs=unit_region.crs
        )

    merged = unit_region.union_all()
    comps = gpd.GeoDataFrame(geometry=[merged], crs=unit_region.crs)
    comps = comps.explode(ignore_index=True, index_parts=False)

    if fill_component_holes:
        comps["geometry"] = comps["geometry"].apply(fill_holes)

    comps["component_id"] = [f"{region_label}_c{i+1}" for i in range(len(comps))]
    return comps[["component_id", "geometry"]].copy()


def assign_units_to_spatial_components(
    unit_region: gpd.GeoDataFrame,
    components: gpd.GeoDataFrame
) -> pd.DataFrame:
    """
    将单元映射到连续空间子区
    """
    if len(unit_region) == 0 or len(components) == 0:
        return pd.DataFrame(columns=["id", "component_id"])

    joined = gpd.sjoin(
        unit_region[["id", "geometry"]],
        components[["component_id", "geometry"]],
        how="inner",
        predicate="intersects"
    )

    joined = joined[["id", "component_id"]].drop_duplicates().copy()
    joined.index = range(len(joined))
    return joined


# =========================================================
# 3. Moran hotspot detection
# =========================================================
def identify_inflow_core(
    od: pd.DataFrame,
    unit: gpd.GeoDataFrame,
    crit_value: float = 0.05,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
) -> gpd.GeoDataFrame:
    """
    Identify the single dominant commuting-inflow centre in the current
    functional region.

    Procedure:
    1. Calculate destination inflow density within the current region.
    2. Identify statistically significant High-High units using Local Moran's I.
    3. Merge spatially contiguous High-High units into hotspot clusters.
    4. Retain only the hotspot cluster with the largest total inflow.

    An empty GeoDataFrame is returned when no significant High-High cluster
    can be identified. No cumulative-inflow threshold is used.
    """
    if len(unit) == 0 or len(od) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    d = od.groupby("d_id", as_index=False)["flow"].sum()
    d.rename(columns={"d_id": "id", "flow": "inflow"}, inplace=True)

    unit2 = pd.merge(unit, d, on="id", how="left")
    unit2["inflow"] = unit2["inflow"].fillna(0.0)

    valid_area = unit2["area"].notna() & (unit2["area"] > 0)
    if not valid_area.all():
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit2["inflow_den"] = unit2["inflow"] / unit2["area"]

    try:
        if use_queen:
            w = lps.weights.Queen.from_dataframe(
                unit2,
                use_index=True,
                silence_warnings=True,
            )
        else:
            w = lps.weights.Rook.from_dataframe(
                unit2,
                use_index=True,
                silence_warnings=True,
            )

        # Local Moran's I is not meaningful when the spatial weights contain
        # no links or the inflow-density surface has no variation.
        if w.n == 0 or sum(len(v) for v in w.neighbors.values()) == 0:
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)
        if np.isclose(unit2["inflow_den"].var(ddof=0), 0.0):
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

        lm = esda.Moran_Local(
            unit2["inflow_den"],
            w,
            transformation="r",
            permutations=999,
            n_jobs=-1,
            seed=42,
        )
        unit2["lisa"] = lm.get_cluster_labels(crit_value=crit_value)
    except Exception:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh = unit2.loc[unit2["lisa"] == "High-High"].copy()
    if len(unit_hh) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh.index = range(len(unit_hh))

    clusters = gpd.GeoDataFrame(geometry=[unit_hh.union_all()], crs=unit.crs)
    clusters = clusters.explode(ignore_index=True, index_parts=False)

    if fill_cluster_holes:
        clusters["geometry"] = clusters["geometry"].apply(fill_holes)

    clusters["cid"] = clusters.index + 1

    hhc = gpd.overlay(
        unit_hh,
        clusters,
        how="intersection",
        keep_geom_type=True,
    )
    hhcg = hhc.groupby("cid", as_index=False)[["inflow", "area"]].sum()

    clusters = pd.merge(clusters, hhcg, on="cid", how="inner")
    if len(clusters) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    # Scale-consistent rule: one functional region -> one dominant centre.
    clusters = (
        clusters.sort_values("inflow", ascending=False, ignore_index=True)
        .head(1)
        .copy()
    )
    clusters.drop(columns=["cid"], inplace=True, errors="ignore")
    clusters.index = range(len(clusters))

    return gpd.GeoDataFrame(clusters, geometry="geometry", crs=unit.crs)


# =========================================================
# 4. Graph helpers
# =========================================================
def induced_od(od: pd.DataFrame, node_ids: List[str]) -> pd.DataFrame:
    node_set = set(node_ids)
    out = od.loc[od["o_id"].isin(node_set) & od["d_id"].isin(node_set)].copy()
    out.index = range(len(out))
    return out


def build_graph_from_od(od_sub: pd.DataFrame) -> Optional[ig.Graph]:
    if len(od_sub) == 0:
        return None

    nodes = pd.unique(pd.concat([od_sub["o_id"], od_sub["d_id"]], axis=0))
    if len(nodes) < 2:
        return None

    G = ig.Graph.DataFrame(
        od_sub[["o_id", "d_id", "flow"]],
        directed=True,
        use_vids=False
    )
    G.es["weight"] = od_sub["flow"].tolist()
    return G


# =========================================================
# 5. Fixed four-level hotspot + consensus community hierarchy
# =========================================================
MAX_CENTER_LEVEL = 3
LOCAL_LEVEL = 4


def recursive_hotspot_community(
    od_all: pd.DataFrame,
    unit_all: gpd.GeoDataFrame,
    node_ids: List[str],
    crit_value: float,
    use_queen: bool,
    fill_cluster_holes: bool,
    n_runs: int,
    tol: float,
    resolution: float,
    level: int,
    path_prefix: str,
    unit_labels: pd.DataFrame,
    region_polygons: Dict[int, List[gpd.GeoDataFrame]],
    diagnostics: List[dict],
):
    """
    Build a fixed four-level analytical hierarchy.

    L1-L3 are centre levels. At every scale, each current functional region is
    represented by at most one centre: the significant High-High hotspot
    cluster with the largest total commuting inflow.

    After identifying an L1 or L2 centre, the centre is removed and the
    residual commuting network is partitioned with consensus Leiden. Each
    spatially connected community is passed to the next centre level.

    After L3 centre identification, recursion stops by design. All units not
    assigned to L1-L3 are assigned to L4 in the main pipeline. There is no
    cumulative-inflow threshold and no minimum-region-size stopping rule.
    """
    if level > MAX_CENTER_LEVEL:
        return

    current_path = path_prefix if path_prefix else "ROOT"
    unit_sub = unit_all.loc[unit_all["id"].isin(node_ids)].copy()
    unit_sub.index = range(len(unit_sub))
    od_sub = induced_od(od_all, node_ids)

    if len(unit_sub) == 0:
        return

    # -------------------------------------------------
    # Step 1. Identify one dominant significant centre
    # -------------------------------------------------
    centers = identify_inflow_core(
        od_sub,
        unit_sub,
        crit_value=crit_value,
        use_queen=use_queen,
        fill_cluster_holes=fill_cluster_holes,
    )

    if len(centers) == 0:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "no_significant_hotspot",
        })
        return

    unit_center = gpd.overlay(
        unit_sub[["id", "geometry"]],
        centers[["geometry"]],
        how="intersection",
        keep_geom_type=True,
    )
    center_ids = unit_center["id"].astype(str).unique().tolist()

    if len(center_ids) == 0:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "hotspot_without_units",
        })
        return

    # Assign only units that have not already received a centre level.
    mask = unit_labels["id"].isin(center_ids)
    unit_labels.loc[
        mask & unit_labels["level"].isna(),
        "level",
    ] = level

    rem_ids = [x for x in node_ids if x not in set(center_ids)]

    # -------------------------------------------------
    # Step 2. Fixed analytical depth: L3 is the last centre level
    # -------------------------------------------------
    if level == MAX_CENTER_LEVEL:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "fixed_depth_reached",
        })
        return

    if len(rem_ids) == 0:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": 0,
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "all_units_in_center",
        })
        return

    # -------------------------------------------------
    # Step 3. Partition the residual network for the next scale
    # -------------------------------------------------
    od_rem = induced_od(od_all, rem_ids)
    G = build_graph_from_od(od_rem)

    if G is None:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "residual_graph_unavailable",
        })
        return

    res = iterative_consensus_leiden(
        G,
        partition_type=la.ModularityVertexPartition,
        resolution=resolution,
        weights="weight",
        n_runs=n_runs,
        tol=tol,
        metric="NMI",
        max_iter=10,
    )

    membership = res["membership"]
    n_comms = len(set(membership)) if membership else 0

    # If Leiden returns only one community, the whole residual region is the
    # functional region at the next scale. No arbitrary size rule is applied.
    if n_comms < 2:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": n_comms,
            "n_spatial_components": 1,
            "continue_split": True,
            "selection_mode": "top1",
            "reason": "single_community_continue",
        })

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=rem_ids,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            level=level + 1,
            path_prefix=path_prefix,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )
        return

    # -------------------------------------------------
    # Step 4. Multiple functional communities
    # -------------------------------------------------
    name_to_comm = dict(zip(G.vs["name"], membership))

    rem_df = pd.DataFrame({
        "id": list(name_to_comm.keys()),
        "comm_id": [name_to_comm[x] for x in name_to_comm.keys()],
    })

    if path_prefix == "":
        rem_df["comm_label"] = rem_df["comm_id"].astype(int).astype(str)
    else:
        rem_df["comm_label"] = (
            path_prefix + "_" + rem_df["comm_id"].astype(int).astype(str)
        )

    region_col = f"region_{level}"
    if region_col not in unit_labels.columns:
        unit_labels[region_col] = np.nan

    component_records = []
    polygon_records = []
    n_components_total = 0

    for comm_label in rem_df["comm_label"].dropna().unique().tolist():
        comm_ids = rem_df.loc[
            rem_df["comm_label"] == comm_label,
            "id",
        ].astype(str).tolist()

        unit_comm = unit_all.loc[
            unit_all["id"].isin(comm_ids),
            ["id", "geometry"],
        ].copy()
        unit_comm.index = range(len(unit_comm))

        comps = split_region_into_spatial_components(
            unit_comm,
            region_label=comm_label,
            fill_component_holes=False,
        )
        if len(comps) == 0:
            continue

        assign_df = assign_units_to_spatial_components(unit_comm, comps)
        if len(assign_df) == 0:
            continue

        n_components_total += len(comps)
        component_records.append(assign_df)

        comp_poly = comps.copy()
        comp_poly.rename(columns={"component_id": "region_id"}, inplace=True)
        comp_poly["level"] = level
        polygon_records.append(comp_poly)

    diagnostics.append({
        "path": current_path,
        "level": level,
        "n_units": len(node_ids),
        "n_centers": len(center_ids),
        "n_remaining": len(rem_ids),
        "n_comms": n_comms,
        "n_spatial_components": n_components_total,
        "continue_split": n_components_total >= 1,
        "selection_mode": "top1",
        "reason": "split_into_functional_communities",
    })

    if len(component_records) == 0:
        return

    rem_component_df = pd.concat(component_records, axis=0, ignore_index=True)
    rem_component_df = rem_component_df.rename(
        columns={"component_id": region_col}
    )

    mapping = dict(zip(rem_component_df["id"], rem_component_df[region_col]))
    unit_labels[region_col] = (
        unit_labels["id"].map(mapping).combine_first(unit_labels[region_col])
    )

    # Only two rounds of community partitioning are needed to produce the
    # functional regions within which L2 and L3 centres are identified.
    if level <= 2 and len(polygon_records) > 0:
        poly = pd.concat(polygon_records, axis=0, ignore_index=True)
        poly = gpd.GeoDataFrame(poly, geometry="geometry", crs=unit_all.crs)
        region_polygons.setdefault(level, []).append(poly)

    child_regions = rem_component_df[region_col].dropna().unique().tolist()
    for rg in child_regions:
        child_ids = rem_component_df.loc[
            rem_component_df[region_col] == rg,
            "id",
        ].astype(str).tolist()

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=child_ids,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            level=level + 1,
            path_prefix=rg,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )


# =========================================================
# 6. Main pipeline
# =========================================================
def detect_hotspot_community_hierarchy(
    unit: gpd.GeoDataFrame,
    od: pd.DataFrame,
    crit_value: float = 0.05,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
    n_runs: int = 100,
    tol: float = 0.01,
    resolution: float = 1.0,
):
    """
    Detect a fixed four-level urban hierarchy.

    Levels:
        L1: metropolitan primary centre
        L2: dominant sub-centre in each first-order functional community
        L3: dominant community centre in each second-order functional community
        L4: all remaining local-level units

    Returns:
        unit_result:
            Input spatial units with fixed integer levels 1-4 and available
            functional-region labels (region_1 and region_2).
        region_polygons_all:
            Spatial polygons of the functional communities generated after
            removing L1 and L2 centres.
        diagnostics_df:
            Branch-level diagnostics for centre identification and community
            partitioning.
    """
    required_unit_cols = {"id", "geometry"}
    required_od_cols = {"o_id", "d_id", "flow"}

    missing_unit = required_unit_cols.difference(unit.columns)
    missing_od = required_od_cols.difference(od.columns)
    if missing_unit:
        raise ValueError(f"unit is missing required columns: {sorted(missing_unit)}")
    if missing_od:
        raise ValueError(f"od is missing required columns: {sorted(missing_od)}")

    unit = unit.copy()
    od = od.copy()

    unit["id"] = unit["id"].astype(str)
    od["o_id"] = od["o_id"].astype(str)
    od["d_id"] = od["d_id"].astype(str)
    od["flow"] = pd.to_numeric(od["flow"], errors="coerce").fillna(0.0)
    od = od.loc[od["flow"] > 0].copy()

    if "area" not in unit.columns:
        unit["area"] = unit.geometry.area / 1e6

    keep_ids = pd.unique(pd.concat([od["o_id"], od["d_id"]], axis=0))
    unit = unit.loc[unit["id"].isin(keep_ids)].copy()
    unit.index = range(len(unit))

    unit_labels = unit[["id"]].copy()
    unit_labels["level"] = np.nan

    region_polygons: Dict[int, List[gpd.GeoDataFrame]] = {}
    diagnostics: List[dict] = []

    root_ids = sorted(unit["id"].unique().tolist())
    recursive_hotspot_community(
        od_all=od,
        unit_all=unit,
        node_ids=root_ids,
        crit_value=crit_value,
        use_queen=use_queen,
        fill_cluster_holes=fill_cluster_holes,
        n_runs=n_runs,
        tol=tol,
        resolution=resolution,
        level=1,
        path_prefix="",
        unit_labels=unit_labels,
        region_polygons=region_polygons,
        diagnostics=diagnostics,
    )

    unit_result = pd.merge(unit, unit_labels, on="id", how="left")

    # The analytical framework always has exactly four levels. Any spatial
    # unit not identified as a significant L1-L3 centre is a local-level unit.
    unit_result["level"] = unit_result["level"].fillna(LOCAL_LEVEL).astype(int)

    region_polygons_all: Dict[int, gpd.GeoDataFrame] = {}
    for lv, polys in region_polygons.items():
        if len(polys) == 0:
            continue
        g = pd.concat(polys, axis=0, ignore_index=True)
        region_polygons_all[lv] = gpd.GeoDataFrame(
            g,
            geometry="geometry",
            crs=unit.crs,
        )

    diagnostics_df = pd.DataFrame(diagnostics)
    return unit_result, region_polygons_all, diagnostics_df


# =========================================================
# 7. Save outputs
# =========================================================
def save_hotspot_community_outputs(
    unit_result: gpd.GeoDataFrame,
    region_polygons_all: Dict[int, gpd.GeoDataFrame],
    diagnostics_df: pd.DataFrame,
    out_dir: str,
    prefix: str
):
    os.makedirs(out_dir, exist_ok=True)

    unit_result.to_file(os.path.join(out_dir, f"{prefix}_unit_hierarchy.shp"))

    if 1 in region_polygons_all:
        region_polygons_all[1].to_file(os.path.join(out_dir, f"{prefix}_region_level_1.shp"))

    if 2 in region_polygons_all:
        region_polygons_all[2].to_file(os.path.join(out_dir, f"{prefix}_region_level_2.shp"))

    diagnostics_df.to_csv(os.path.join(out_dir, f"{prefix}_diagnostics.csv"), index=False)


# =========================================================
# 8. Example usage
# =========================================================
def get_unit_name(city: str) -> str:
    if city in ["beijing", "shanghai", "shenzhen", "nanjing"]:
        return "grid_1k"
    elif city == "london":
        return "msoa"
    elif city in ["losangeles", "newyork"]:
        return "tract"
    else:
        raise ValueError(f"Unknown city: {city}")

## Final version

In [15]:
import os
import warnings
from typing import Optional, List, Dict, Tuple, Sequence

import numpy as np
import pandas as pd
import geopandas as gpd
import igraph as ig
import leidenalg as la
import libpysal as lps
import esda

from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import make_valid
from sklearn.metrics import normalized_mutual_info_score, fowlkes_mallows_score

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# 1. Consensus Leiden
# =========================================================
def _run_leiden_once(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    seed: int
):
    kwargs = dict(
        weights=weights,
        n_iterations=-1,
        seed=seed
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution
    return la.find_partition(G, partition_type, **kwargs)


def _multi_run_memberships(
    G: ig.Graph,
    partition_type,
    resolution: float,
    weights: Optional[str],
    n_runs: int,
    seed0: int = 0
) -> List[List[int]]:
    membs = []
    for r in range(n_runs):
        part = _run_leiden_once(G, partition_type, resolution, weights, seed0 + r)
        membs.append(list(part.membership))
    return membs


def _coassoc_from_memberships(membs: List[List[int]]) -> np.ndarray:
    n = len(membs[0])
    P = np.zeros((n, n), dtype=np.float64)

    for m in membs:
        buckets: Dict[int, list] = {}
        for idx, c in enumerate(m):
            buckets.setdefault(c, []).append(idx)
        for idxs in buckets.values():
            idxs = np.asarray(idxs, dtype=int)
            P[np.ix_(idxs, idxs)] += 1.0

    P /= float(len(membs))
    np.fill_diagonal(P, 0.0)
    P = 0.5 * (P + P.T)
    return P


def _consensus_on_coassoc(
    P: np.ndarray,
    partition_type,
    resolution: float,
    threshold: Optional[float] = None
) -> Tuple[List[int], ig.Graph]:
    P_use = P.copy()
    if threshold is not None:
        P_use[P_use < threshold] = 0.0

    Gc = ig.Graph.Weighted_Adjacency(
        P_use.tolist(),
        mode="UNDIRECTED",
        attr="weight",
        loops=False
    )

    kwargs = dict(
        weights="weight",
        n_iterations=-1,
        seed=0
    )
    if partition_type != la.ModularityVertexPartition:
        kwargs["resolution_parameter"] = resolution

    part = la.find_partition(Gc, partition_type, **kwargs)
    return list(part.membership), Gc


def _similarity(m1: List[int], m2: List[int], metric: str = "NMI") -> float:
    if metric.upper() == "NMI":
        return normalized_mutual_info_score(m1, m2)
    elif metric.upper() == "FMI":
        return fowlkes_mallows_score(m1, m2)
    else:
        raise ValueError("metric must be 'NMI' or 'FMI'.")


def iterative_consensus_leiden(
    G: ig.Graph,
    partition_type=la.ModularityVertexPartition,
    resolution: float = 1.0,
    weights: Optional[str] = "weight",
    n_runs: int = 100,
    seed0: int = 0,
    max_iter: int = 10,
    tol: float = 0.01,
    metric: str = "NMI",
    threshold: Optional[float] = None,
    return_history: bool = True
):
    if G is None or G.vcount() == 0:
        return {"membership": [], "history": [], "coassoc": None}

    membs0 = _multi_run_memberships(G, partition_type, resolution, weights, n_runs, seed0)
    P = _coassoc_from_memberships(membs0)
    memb_prev, Gc = _consensus_on_coassoc(P, partition_type, resolution, threshold)
    hist = [{"iter": 0, "similarity": np.nan, "n_comms": len(set(memb_prev))}]

    for it in range(1, max_iter + 1):
        membs = _multi_run_memberships(
            Gc,
            partition_type,
            resolution,
            "weight",
            n_runs,
            seed0 + it * 1000,
        )
        P_next = _coassoc_from_memberships(membs)
        memb_next, Gc_next = _consensus_on_coassoc(P_next, partition_type, resolution, threshold)

        sim = _similarity(memb_prev, memb_next, metric=metric)
        hist.append({"iter": it, "similarity": sim, "n_comms": len(set(memb_next))})

        if 1.0 - sim < tol:
            return {
                "membership": memb_next,
                "history": hist if return_history else None,
                "coassoc": P_next
            }

        memb_prev, Gc, P = memb_next, Gc_next, P_next

    return {
        "membership": memb_prev,
        "history": hist if return_history else None,
        "coassoc": P
    }


# =========================================================
# 2. Geometry helpers
# =========================================================
def fill_holes(geom):
    if geom is None:
        return None

    geom = make_valid(geom)

    if geom.geom_type == "Polygon":
        return Polygon(geom.exterior)

    if geom.geom_type == "MultiPolygon":
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])

    return geom


def split_region_into_spatial_components(
    unit_region: gpd.GeoDataFrame,
    region_label: str,
    fill_component_holes: bool = False
) -> gpd.GeoDataFrame:
    """
    Split one functional community into spatially contiguous components.
    """
    if len(unit_region) == 0:
        return gpd.GeoDataFrame(
            columns=["component_id", "geometry"],
            geometry="geometry",
            crs=unit_region.crs
        )

    merged = unit_region.union_all()
    comps = gpd.GeoDataFrame(geometry=[merged], crs=unit_region.crs)
    comps = comps.explode(ignore_index=True, index_parts=False)

    if fill_component_holes:
        comps["geometry"] = comps["geometry"].apply(fill_holes)

    comps["component_id"] = [f"{region_label}_c{i+1}" for i in range(len(comps))]
    return comps[["component_id", "geometry"]].copy()


def assign_units_to_spatial_components(
    unit_region: gpd.GeoDataFrame,
    components: gpd.GeoDataFrame
) -> pd.DataFrame:
    """
    Assign spatial units to contiguous components.
    """
    if len(unit_region) == 0 or len(components) == 0:
        return pd.DataFrame(columns=["id", "component_id"])

    joined = gpd.sjoin(
        unit_region[["id", "geometry"]],
        components[["component_id", "geometry"]],
        how="inner",
        predicate="intersects"
    )

    joined = joined[["id", "component_id"]].drop_duplicates().copy()
    joined.index = range(len(joined))
    return joined


# =========================================================
# 3. Moran hotspot detection
# =========================================================
def identify_inflow_core(
    od: pd.DataFrame,
    unit: gpd.GeoDataFrame,
    crit_value: float = 0.05,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
) -> gpd.GeoDataFrame:
    """
    Identify the single dominant commuting-inflow centre in the current
    functional region.

    Procedure:
    1. Calculate destination inflow density within the current region.
    2. Identify statistically significant High-High units using Local Moran's I.
    3. Merge spatially contiguous High-High units into hotspot clusters.
    4. Retain only the hotspot cluster with the largest total inflow.

    An empty GeoDataFrame is returned when no significant High-High cluster
    can be identified. No cumulative-inflow threshold is used.
    """
    if len(unit) == 0 or len(od) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    d = od.groupby("d_id", as_index=False)["flow"].sum()
    d.rename(columns={"d_id": "id", "flow": "inflow"}, inplace=True)

    unit2 = pd.merge(unit, d, on="id", how="left")
    unit2["inflow"] = unit2["inflow"].fillna(0.0)

    valid_area = unit2["area"].notna() & (unit2["area"] > 0)
    if not valid_area.all():
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit2["inflow_den"] = unit2["inflow"] / unit2["area"]

    try:
        if use_queen:
            w = lps.weights.Queen.from_dataframe(
                unit2,
                use_index=True,
                silence_warnings=True,
            )
        else:
            w = lps.weights.Rook.from_dataframe(
                unit2,
                use_index=True,
                silence_warnings=True,
            )

        # Local Moran's I is not meaningful when the spatial weights contain
        # no links or the inflow-density surface has no variation.
        if w.n == 0 or sum(len(v) for v in w.neighbors.values()) == 0:
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)
        if np.isclose(unit2["inflow_den"].var(ddof=0), 0.0):
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

        lm = esda.Moran_Local(
            unit2["inflow_den"],
            w,
            transformation="r",
            permutations=999,
            n_jobs=-1,
            seed=42,
        )
        unit2["lisa"] = lm.get_cluster_labels(crit_value=crit_value)
    except Exception:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh = unit2.loc[unit2["lisa"] == "High-High"].copy()
    if len(unit_hh) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    unit_hh.index = range(len(unit_hh))

    clusters = gpd.GeoDataFrame(geometry=[unit_hh.union_all()], crs=unit.crs)
    clusters = clusters.explode(ignore_index=True, index_parts=False)

    if fill_cluster_holes:
        clusters["geometry"] = clusters["geometry"].apply(fill_holes)

    clusters["cid"] = clusters.index + 1

    hhc = gpd.overlay(
        unit_hh,
        clusters,
        how="intersection",
        keep_geom_type=True,
    )
    hhcg = hhc.groupby("cid", as_index=False)[["inflow", "area"]].sum()

    clusters = pd.merge(clusters, hhcg, on="cid", how="inner")
    if len(clusters) == 0:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=unit.crs)

    # Scale-consistent rule: one functional region -> one dominant centre.
    clusters = (
        clusters.sort_values("inflow", ascending=False, ignore_index=True)
        .head(1)
        .copy()
    )
    clusters.drop(columns=["cid"], inplace=True, errors="ignore")
    clusters.index = range(len(clusters))

    return gpd.GeoDataFrame(clusters, geometry="geometry", crs=unit.crs)


# =========================================================
# 4. Graph helpers
# =========================================================
def induced_od(od: pd.DataFrame, node_ids: List[str]) -> pd.DataFrame:
    node_set = set(node_ids)
    out = od.loc[od["o_id"].isin(node_set) & od["d_id"].isin(node_set)].copy()
    out.index = range(len(out))
    return out


def build_graph_from_od(od_sub: pd.DataFrame) -> Optional[ig.Graph]:
    if len(od_sub) == 0:
        return None

    nodes = pd.unique(pd.concat([od_sub["o_id"], od_sub["d_id"]], axis=0))
    if len(nodes) < 2:
        return None

    G = ig.Graph.DataFrame(
        od_sub[["o_id", "d_id", "flow"]],
        directed=True,
        use_vids=False
    )
    G.es["weight"] = od_sub["flow"].tolist()
    return G


# =========================================================
# 5. Configurable hotspot + consensus community hierarchy
# =========================================================
def recursive_hotspot_community(
    od_all: pd.DataFrame,
    unit_all: gpd.GeoDataFrame,
    node_ids: List[str],
    crit_value: float,
    use_queen: bool,
    fill_cluster_holes: bool,
    n_runs: int,
    tol: float,
    resolution: float,
    level: int,
    max_center_level: int,
    path_prefix: str,
    unit_labels: pd.DataFrame,
    region_polygons: Dict[int, List[gpd.GeoDataFrame]],
    diagnostics: List[dict],
):
    """
    Recursively identify centres until the requested analytical depth.

    Levels 1 to ``max_center_level`` are centre levels. At every scale, each
    current functional region is represented by at most one centre: the
    significant High-High hotspot cluster with the largest total commuting
    inflow.

    After identifying a centre, it is removed and the residual commuting
    network is partitioned with consensus Leiden. Each spatially connected
    community is passed to the next centre level.

    Recursion stops after ``max_center_level``. Units not assigned to a centre
    level are assigned to the final residual level in the main pipeline.
    """
    if level > max_center_level:
        return

    current_path = path_prefix if path_prefix else "ROOT"
    unit_sub = unit_all.loc[unit_all["id"].isin(node_ids)].copy()
    unit_sub.index = range(len(unit_sub))
    od_sub = induced_od(od_all, node_ids)

    if len(unit_sub) == 0:
        return

    # -------------------------------------------------
    # Step 1. Identify one dominant significant centre
    # -------------------------------------------------
    centers = identify_inflow_core(
        od_sub,
        unit_sub,
        crit_value=crit_value,
        use_queen=use_queen,
        fill_cluster_holes=fill_cluster_holes,
    )

    if len(centers) == 0:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "no_significant_hotspot",
        })
        return

    unit_center = gpd.overlay(
        unit_sub[["id", "geometry"]],
        centers[["geometry"]],
        how="intersection",
        keep_geom_type=True,
    )
    center_ids = unit_center["id"].astype(str).unique().tolist()

    if len(center_ids) == 0:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": 0,
            "n_remaining": len(node_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "hotspot_without_units",
        })
        return

    # Assign only units that have not already received a centre level.
    mask = unit_labels["id"].isin(center_ids)
    unit_labels.loc[
        mask & unit_labels["level"].isna(),
        "level",
    ] = level

    rem_ids = [x for x in node_ids if x not in set(center_ids)]

    # -------------------------------------------------
    # Step 2. Stop after the requested final centre level
    # -------------------------------------------------
    if level == max_center_level:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "maximum_center_level_reached",
        })
        return

    if len(rem_ids) == 0:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": 0,
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "all_units_in_center",
        })
        return

    # -------------------------------------------------
    # Step 3. Partition the residual network for the next scale
    # -------------------------------------------------
    od_rem = induced_od(od_all, rem_ids)
    G = build_graph_from_od(od_rem)

    if G is None:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": 0,
            "n_spatial_components": 0,
            "continue_split": False,
            "selection_mode": "top1",
            "reason": "residual_graph_unavailable",
        })
        return

    res = iterative_consensus_leiden(
        G,
        partition_type=la.ModularityVertexPartition,
        resolution=resolution,
        weights="weight",
        n_runs=n_runs,
        tol=tol,
        metric="NMI",
        max_iter=10,
    )

    membership = res["membership"]
    n_comms = len(set(membership)) if membership else 0

    # If Leiden returns only one community, the whole residual region is the
    # functional region at the next scale. No arbitrary size rule is applied.
    if n_comms < 2:
        diagnostics.append({
            "path": current_path,
            "level": level,
            "n_units": len(node_ids),
            "n_centers": len(center_ids),
            "n_remaining": len(rem_ids),
            "n_comms": n_comms,
            "n_spatial_components": 1,
            "continue_split": True,
            "selection_mode": "top1",
            "reason": "single_community_continue",
        })

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=rem_ids,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            level=level + 1,
            max_center_level=max_center_level,
            path_prefix=path_prefix,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )
        return

    # -------------------------------------------------
    # Step 4. Multiple functional communities
    # -------------------------------------------------
    name_to_comm = dict(zip(G.vs["name"], membership))

    rem_df = pd.DataFrame({
        "id": list(name_to_comm.keys()),
        "comm_id": [name_to_comm[x] for x in name_to_comm.keys()],
    })

    if path_prefix == "":
        rem_df["comm_label"] = rem_df["comm_id"].astype(int).astype(str)
    else:
        rem_df["comm_label"] = (
            path_prefix + "_" + rem_df["comm_id"].astype(int).astype(str)
        )

    region_col = f"region_{level}"
    if region_col not in unit_labels.columns:
        unit_labels[region_col] = np.nan

    component_records = []
    polygon_records = []
    n_components_total = 0

    for comm_label in rem_df["comm_label"].dropna().unique().tolist():
        comm_ids = rem_df.loc[
            rem_df["comm_label"] == comm_label,
            "id",
        ].astype(str).tolist()

        unit_comm = unit_all.loc[
            unit_all["id"].isin(comm_ids),
            ["id", "geometry"],
        ].copy()
        unit_comm.index = range(len(unit_comm))

        comps = split_region_into_spatial_components(
            unit_comm,
            region_label=comm_label,
            fill_component_holes=False,
        )
        if len(comps) == 0:
            continue

        assign_df = assign_units_to_spatial_components(unit_comm, comps)
        if len(assign_df) == 0:
            continue

        n_components_total += len(comps)
        component_records.append(assign_df)

        comp_poly = comps.copy()
        comp_poly.rename(columns={"component_id": "region_id"}, inplace=True)
        comp_poly["level"] = level
        polygon_records.append(comp_poly)

    diagnostics.append({
        "path": current_path,
        "level": level,
        "n_units": len(node_ids),
        "n_centers": len(center_ids),
        "n_remaining": len(rem_ids),
        "n_comms": n_comms,
        "n_spatial_components": n_components_total,
        "continue_split": n_components_total >= 1,
        "selection_mode": "top1",
        "reason": "split_into_functional_communities",
    })

    if len(component_records) == 0:
        return

    rem_component_df = pd.concat(component_records, axis=0, ignore_index=True)
    rem_component_df = rem_component_df.rename(
        columns={"component_id": region_col}
    )

    mapping = dict(zip(rem_component_df["id"], rem_component_df[region_col]))
    unit_labels[region_col] = (
        unit_labels["id"].map(mapping).combine_first(unit_labels[region_col])
    )

    # Store every residual partition that defines regions for the next centre
    # level. A K-level solution therefore stores region levels 1 to K-2.
    if level < max_center_level and len(polygon_records) > 0:
        poly = pd.concat(polygon_records, axis=0, ignore_index=True)
        poly = gpd.GeoDataFrame(poly, geometry="geometry", crs=unit_all.crs)
        region_polygons.setdefault(level, []).append(poly)

    child_regions = rem_component_df[region_col].dropna().unique().tolist()
    for rg in child_regions:
        child_ids = rem_component_df.loc[
            rem_component_df[region_col] == rg,
            "id",
        ].astype(str).tolist()

        recursive_hotspot_community(
            od_all=od_all,
            unit_all=unit_all,
            node_ids=child_ids,
            crit_value=crit_value,
            use_queen=use_queen,
            fill_cluster_holes=fill_cluster_holes,
            n_runs=n_runs,
            tol=tol,
            resolution=resolution,
            level=level + 1,
            max_center_level=max_center_level,
            path_prefix=rg,
            unit_labels=unit_labels,
            region_polygons=region_polygons,
            diagnostics=diagnostics,
        )


# =========================================================
# 6. Main pipeline
# =========================================================
def detect_hotspot_community_hierarchy(
    unit: gpd.GeoDataFrame,
    od: pd.DataFrame,
    n_levels: int = 4,
    crit_value: float = 0.05,
    use_queen: bool = True,
    fill_cluster_holes: bool = True,
    n_runs: int = 100,
    tol: float = 0.01,
    resolution: float = 1.0,
):
    """
    Detect a three-, four-, or five-level urban hierarchy.

    The first ``n_levels - 1`` levels contain the dominant significant centre
    identified within each functional region. The final level contains all
    units not assigned to a centre level.

    For example, with ``n_levels=4``, L1-L3 are centre levels and L4 is the
    residual non-centre level.

    Returns:
        unit_result:
            Input spatial units with integer levels from 1 to ``n_levels`` and
            all available functional-region labels.
        region_polygons_all:
            Spatial polygons of the functional communities generated before
            each subsequent centre level.
        diagnostics_df:
            Branch-level diagnostics for centre identification and community
            partitioning.
    """
    if n_levels not in (3, 4, 5):
        raise ValueError("n_levels must be 3, 4, or 5.")

    max_center_level = n_levels - 1

    required_unit_cols = {"id", "geometry"}
    required_od_cols = {"o_id", "d_id", "flow"}

    missing_unit = required_unit_cols.difference(unit.columns)
    missing_od = required_od_cols.difference(od.columns)
    if missing_unit:
        raise ValueError(f"unit is missing required columns: {sorted(missing_unit)}")
    if missing_od:
        raise ValueError(f"od is missing required columns: {sorted(missing_od)}")

    unit = unit.copy()
    od = od.copy()

    unit["id"] = unit["id"].astype(str)
    od["o_id"] = od["o_id"].astype(str)
    od["d_id"] = od["d_id"].astype(str)
    od["flow"] = pd.to_numeric(od["flow"], errors="coerce").fillna(0.0)
    od = od.loc[od["flow"] > 0].copy()

    if "area" not in unit.columns:
        unit["area"] = unit.geometry.area / 1e6

    keep_ids = pd.unique(pd.concat([od["o_id"], od["d_id"]], axis=0))
    unit = unit.loc[unit["id"].isin(keep_ids)].copy()
    unit.index = range(len(unit))

    unit_labels = unit[["id"]].copy()
    unit_labels["level"] = np.nan

    region_polygons: Dict[int, List[gpd.GeoDataFrame]] = {}
    diagnostics: List[dict] = []

    root_ids = sorted(unit["id"].unique().tolist())
    recursive_hotspot_community(
        od_all=od,
        unit_all=unit,
        node_ids=root_ids,
        crit_value=crit_value,
        use_queen=use_queen,
        fill_cluster_holes=fill_cluster_holes,
        n_runs=n_runs,
        tol=tol,
        resolution=resolution,
        level=1,
        max_center_level=max_center_level,
        path_prefix="",
        unit_labels=unit_labels,
        region_polygons=region_polygons,
        diagnostics=diagnostics,
    )

    unit_result = pd.merge(unit, unit_labels, on="id", how="left")

    # Any unit not assigned to one of the centre levels belongs to the final
    # residual non-centre level.
    unit_result["level"] = unit_result["level"].fillna(n_levels).astype(int)

    region_polygons_all: Dict[int, gpd.GeoDataFrame] = {}
    for lv, polys in region_polygons.items():
        if len(polys) == 0:
            continue
        g = pd.concat(polys, axis=0, ignore_index=True)
        region_polygons_all[lv] = gpd.GeoDataFrame(
            g,
            geometry="geometry",
            crs=unit.crs,
        )

    diagnostics_df = pd.DataFrame(diagnostics)
    if len(diagnostics_df) > 0:
        diagnostics_df.insert(0, "n_levels", n_levels)
    return unit_result, region_polygons_all, diagnostics_df


def detect_hierarchy_depths(
    unit: gpd.GeoDataFrame,
    od: pd.DataFrame,
    n_levels_options: Sequence[int] = (3, 4, 5),
    **kwargs,
) -> Dict[int, Tuple[gpd.GeoDataFrame, Dict[int, gpd.GeoDataFrame], pd.DataFrame]]:
    """Run the same hierarchy method for multiple analytical depths."""
    options = tuple(dict.fromkeys(n_levels_options))
    invalid = [n for n in options if n not in (3, 4, 5)]
    if invalid:
        raise ValueError(f"Unsupported hierarchy depths: {invalid}")

    return {
        n_levels: detect_hotspot_community_hierarchy(
            unit=unit,
            od=od,
            n_levels=n_levels,
            **kwargs,
        )
        for n_levels in options
    }


# =========================================================
# 7. Save outputs
# =========================================================
def save_hotspot_community_outputs(
    unit_result: gpd.GeoDataFrame,
    region_polygons_all: Dict[int, gpd.GeoDataFrame],
    diagnostics_df: pd.DataFrame,
    out_dir: str,
    prefix: str
):
    os.makedirs(out_dir, exist_ok=True)

    unit_result.to_file(os.path.join(out_dir, f"{prefix}_hierarchy.shp"))

    for level, polygons in sorted(region_polygons_all.items()):
        polygons.to_file(
            os.path.join(out_dir, f"{prefix}_region_level_{level}.shp")
        )

    diagnostics_df.to_csv(os.path.join(out_dir, f"{prefix}_diagnostics.csv"), index=False)


def save_hierarchy_depth_outputs(
    results_by_depth: Dict[
        int,
        Tuple[gpd.GeoDataFrame, Dict[int, gpd.GeoDataFrame], pd.DataFrame],
    ],
    out_dir: str,
    prefix: str,
):
    """Save each hierarchy depth in a separate subdirectory."""
    for n_levels, result in sorted(results_by_depth.items()):
        unit_result, region_polygons_all, diagnostics_df = result
        depth_dir = os.path.join(out_dir, f"{n_levels}_levels")
        save_hotspot_community_outputs(
            unit_result=unit_result,
            region_polygons_all=region_polygons_all,
            diagnostics_df=diagnostics_df,
            out_dir=depth_dir,
            prefix=prefix,
        )


# =========================================================
# 8. Example usage
# =========================================================
def get_unit_name(city: str) -> str:
    if city in ["beijing", "shanghai", "shenzhen", "nanjing"]:
        return "grid_1k"
    elif city == "london":
        return "msoa"
    elif city in ["losangeles", "newyork"]:
        return "tract"
    else:
        raise ValueError(f"Unknown city: {city}")

In [17]:
for city in tqdm(["beijing","shanghai","shenzhen","london","losangeles","newyork"]):
    if city in ["beijing","shanghai","shenzhen"]:
        use_queen = False
    else:
        use_queen = True

    unit_name = get_unit_name(city)

    od = pd.read_csv(f"E:/commuting_flow_models/od_tables/{city}_od.csv")
    od = od.loc[(od["o_id"] != od["d_id"]) & (od["flow"] > 0)].copy()
    od.index = range(len(od))

    unit = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{unit_name}.shp")
    unit["area"] = unit.geometry.area / 1e6
    unit = unit[["id", "area", "geometry"]].copy()
    unit = unit.loc[unit['id'].isin(od['o_id']) | unit['id'].isin(od['d_id'])].copy()
    unit.index = range(len(unit))

    results_by_depth = detect_hierarchy_depths(
        unit=unit,
        od=od,
        n_levels_options=(5,),
        crit_value=0.05,
        use_queen=use_queen,
        n_runs=100,
        tol=0.01,
        resolution=1.0,
    )

    save_hierarchy_depth_outputs(
        results_by_depth=results_by_depth,
        out_dir="D:/urban_hierarchy_congestion/results/hierarchy_identification_results",
        prefix=city,
    )

100%|██████████| 6/6 [1:07:49<00:00, 678.31s/it]


## Aggregate the number of POIs by spatial unit

In [7]:
city = 'beijing'
poi = gpd.read_file(f'D:/urban_hierarchy_congestion/data/poi/{city}_poi.shp')

In [8]:
poi['type'].unique()

array(['food', 'company', 'industry', 'retail', 'bus_stop', 'metro_stop',
       'transport_hub', 'education', 'residence', 'health', 'government',
       'accommodation'], dtype=object)

In [6]:
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    poi = gpd.read_file(f'E:/commuting_flow_models/studyarea/{city}/{city}_poi.shp')
    poi.to_file(f'D:/urban_hierarchy_congestion/data/poi/{city}_poi.shp')

100%|██████████| 6/6 [02:21<00:00, 23.64s/it]


In [19]:
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    poi = gpd.read_file(f'D:/urban_hierarchy_congestion/data/poi/{city}_poi.shp')
    unit = gpd.read_file(f'D:/urban_hierarchy_congestion/results/hierarchy_identification_results/5_levels/{city}_hierarchy.shp')
    poi.to_crs(crs=unit.crs,inplace=True)

    for i in ['residence','company','food','retail','accommodation','education','health','government','industry','transport_hub']:
        unit_poi = gpd.sjoin(left_df=unit,right_df=poi.loc[(poi['type']==i),['type','geometry']],predicate='contains').groupby(['id'],as_index=False).size()
        unit = pd.merge(unit,unit_poi,on='id',how='left')
        unit.rename(columns={'size':i[:3]},inplace=True)
        unit.fillna(0,inplace=True)
    unit.to_file(f"D:/urban_hierarchy_congestion/results/hierarchy_identification_results/5_levels/{city}_hierarchy_poi.shp")

100%|██████████| 6/6 [01:02<00:00, 10.33s/it]


In [33]:
from sklearn.metrics import roc_auc_score
city = 'losangeles'
unit = gpd.read_file(f"D:/urban_hierarchy_congestion/results/hierarchy_identification_results/5_levels/{city}_hierarchy_poi.shp")
unit['poi_den'] = (unit['com']+unit['foo']+unit['acc']+unit['edu']+unit['hea']+unit['gov']+unit['ind']+unit['tra'])/unit['area']

In [46]:
unit['region_1'].unique()

array(['2_c1', '0', '0_c1', '5_c1', '4_c1', '1_c1', '3_c1', '1_c2'],
      dtype=object)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def get_parent_region(data, center_level):
    """Return the deepest available parent-region label."""

    if center_level == 1:
        return pd.Series("CITY", index=data.index)

    candidate_columns = [
        f"region_{i}" for i in range(center_level - 1, 0, -1)
    ]

    parent_region = pd.Series(pd.NA, index=data.index, dtype="object")

    for column in candidate_columns:
        available = parent_region.isna() & data[column].notna()
        parent_region.loc[available] = (
            column + ":" + data.loc[available, column].astype(str)
        )

    return parent_region.fillna("ROOT")


def calculate_local_poi_auc(data):
    """
    Calculate local POI-density AUC for L1-L4 centers identified
    from a five-level hierarchy.
    """

    required_columns = {
        "level",
        "region_1",
        "region_2",
        "region_3",
        "poi_den",
    }

    missing_columns = required_columns.difference(data.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    df = data.copy()
    df["level"] = pd.to_numeric(df["level"], errors="coerce")
    df["poi_den"] = pd.to_numeric(df["poi_den"], errors="coerce")

    records = []

    for center_level in range(1, 5):

        # Exclude centers identified at higher levels.
        level_data = df.loc[df["level"] >= center_level].copy()

        level_data["parent_region"] = get_parent_region(
            level_data,
            center_level,
        )

        for parent_region, group in level_data.groupby(
            "parent_region",
            dropna=False,
        ):
            group = group.dropna(subset=["poi_den"]).copy()

            center_mask = group["level"] == center_level
            residual_mask = group["level"] > center_level

            n_center = int(center_mask.sum())
            n_residual = int(residual_mask.sum())

            if n_center > 0 and n_residual > 0:
                center_label = center_mask.astype(int)

                local_auc = roc_auc_score(
                    center_label,
                    group["poi_den"],
                )

                center_poi_median = group.loc[
                    center_mask, "poi_den"
                ].median()

                residual_poi_median = group.loc[
                    residual_mask, "poi_den"
                ].median()
            else:
                local_auc = np.nan
                center_poi_median = np.nan
                residual_poi_median = np.nan

            records.append({
                "center_level": center_level,
                "parent_region": parent_region,
                "n_center_units": n_center,
                "n_residual_units": n_residual,
                "local_auc": local_auc,
                "center_poi_median": center_poi_median,
                "residual_poi_median": residual_poi_median,
            })

    local_auc = pd.DataFrame(records)

    summary = (
        local_auc.groupby("center_level", as_index=False)
        .agg(
            n_parent_regions=("parent_region", "size"),
            n_valid_auc=("local_auc", "count"),
            median_local_auc=("local_auc", "median"),
            mean_local_auc=("local_auc", "mean"),
            q25_local_auc=(
                "local_auc",
                lambda x: x.quantile(0.25),
            ),
            q75_local_auc=(
                "local_auc",
                lambda x: x.quantile(0.75),
            ),
            share_auc_above_05=(
                "local_auc",
                lambda x: 100 * (x.dropna() > 0.5).mean()
                if x.notna().any()
                else np.nan,
            ),
        )
    )

    return local_auc, summary


for city in tqdm(['london']):
    unit = gpd.read_file(f"D:/urban_hierarchy_congestion/results/hierarchy_identification_results/5_levels/{city}_hierarchy_poi.shp")
    unit['poi_den'] = (unit['com']+unit['foo']+unit['acc']+unit['edu']+unit['hea']+unit['gov']+unit['ind']+unit['tra'])/unit['area']

    local_auc, level_summary = calculate_local_poi_auc(unit)

    local_auc.to_csv(
        f"results/summary_tables/{city}_local_poi_auc_by_parent_region.csv",
        index=False,
    )

    level_summary.to_csv(
        f"results/summary_tables/{city}_local_poi_auc_by_level.csv",
        index=False,
    )

100%|██████████| 1/1 [00:00<00:00,  1.20it/s]


In [34]:
results = []

for n_levels in [2, 3, 4, 5]:
    center_label = (unit['level'] < n_levels).astype(int)
    poi_score = unit["poi_den"]
    auc = roc_auc_score(center_label, poi_score)
    results.append({
        "n_levels": n_levels,
        "auc": auc,
        "center_share": center_label.mean(),
        "center_poi_median": unit.loc[
            center_label == 1, "poi_den"
        ].median(),
        "noncenter_poi_median": unit.loc[
            center_label == 0, "poi_den"
        ].median()
    })

results = pd.DataFrame(results)
print(results)

   n_levels       auc  center_share  center_poi_median  noncenter_poi_median
0         2  0.952266      0.020903         747.406687             91.252232
1         3  0.901249      0.037867         537.764012             89.922106
2         4  0.844999      0.078461         329.239383             86.532400
3         5  0.827910      0.116328         250.607427             83.057307


In [48]:
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    unit = gpd.read_file(f'D:/urban_hierarchy_congestion/results/hierarchy_identification_results/5_levels/{city}_hierarchy.shp')
    unit.loc[(unit['level']>=3),'level'] = 3
    unit.drop(columns=['region_2','region_3'],inplace=True)
    unit.to_file(f'D:/urban_hierarchy_congestion/results/hierarchy_identification_results/3_levels/{city}_hierarchy.shp')

100%|██████████| 6/6 [00:00<00:00,  8.03it/s]


In [53]:
city = 'beijing'
unit = gpd.read_file(f'D:/urban_hierarchy_congestion/results/hierarchy_identification_results/5_levels/{city}_hierarchy.shp')
unit.dtypes

id            object
area         float64
level          int32
region_1      object
region_2      object
region_3      object
geometry    geometry
dtype: object

In [55]:
city = 'beijing'
unit = gpd.read_file(f"D:/urban_hierarchy_congestion/results/hierarchy_identification_results/5_levels/{city}_hierarchy.shp")

In [59]:
unit['id'] = unit['id'].astype('int32')

In [61]:
unit.dtypes

id             int32
area         float64
level          int32
region_1      object
region_2      object
region_3      object
geometry    geometry
dtype: object

In [56]:
od = pd.read_csv(f'D:/urban_hierarchy_congestion/data/od_tables/{city}_od.csv')

In [66]:
od2 = pd.merge(od,unit[['id']],left_on='o_id',right_on='id')

# Loubar-based hierarchy

In [ ]:
def compute_lorenz_curve(values):
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cum_nodes = np.arange(1, n + 1) / n
    cum_flows = np.cumsum(sorted_vals) / np.sum(sorted_vals)
    return cum_nodes, cum_flows

def gini(array):
    array = np.sort(np.array(array))
    n = len(array)
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * array)) / (n * np.sum(array))

def assign_hotspot_levels_auto(df,density_column):
    df = df.copy()
    remaining = df[['id', density_column]].copy()

    level = 1
    while remaining[density_column].mean() > 0:
        slope = remaining[density_column].max()/remaining[density_column].mean()

        if slope <= 1.001: 
            break

        threshold_frac = 1 - (1 / slope)
        threshold_frac = np.clip(threshold_frac, 0, 1)

        cutoff = int(np.ceil(len(remaining) * (1 - threshold_frac)))
        if cutoff <= 0:
            break

        top_ids = remaining.nlargest(cutoff, density_column)['id'].values
        df.loc[df['id'].isin(top_ids), 'level'] = level

        remaining = remaining[~remaining['id'].isin(top_ids)]

        level += 1
    
    df.loc[(df['level'].isnull()),'level'] = level
    df['level'] = df['level'].astype(int)

    return df

In [ ]:
def get_unit_name(city: str) -> str:
    if city in ["beijing", "shanghai", "shenzhen", "nanjing"]:
        return "grid_1k"
    elif city == "london":
        return "msoa"
    elif city in ["losangeles", "newyork"]:
        return "tract"
    else:
        raise ValueError(f"Unknown city: {city}")
    
for city in tqdm(['beijing','shanghai','shenzhen','london','losangeles','newyork']):
    unit_name = get_unit_name(city)
    od = pd.read_csv(f'D:/urban_hierarchy_congestion/data/od_tables/{city}_od.csv')
    d = od.groupby('d_id',as_index=False)['flow'].sum()
    d.rename(columns={'d_id':'id','flow':'inflow'},inplace=True)
    unit = gpd.read_file(f'D:/urban_hierarchy_congestion/data/taz/{city}_{unit_name}.shp')
    unit['area'] = unit.geometry.area/1e6
    unit = unit[["id", "area", "geometry"]].copy()
    unit = unit.loc[unit['id'].isin(od['o_id']) | unit['id'].isin(od['d_id'])].copy()
    unit.index = range(len(unit))
    
    unit = pd.merge(unit,d,on='id',how='left')
    unit.fillna(0,inplace=True)
    unit['inflowd'] = unit['inflow']/unit['area']
    unit = assign_hotspot_levels_auto(unit,'inflowd')
    unit.drop(columns=['inflow','inflowd'],inplace=True)
    unit['level'] = unit['level'].astype(int)
    unit.loc[(unit['level']>=4),'level'] = 4
    unit['region_1'] = '0'
    unit['region_2'] = '0'

    poi = gpd.read_file(f'D:/urban_hierarchy_congestion/data/poi/{city}_poi.shp')
    poi.to_crs(crs=unit.crs,inplace=True)

    for i in ['commercial_service','education','health','government','industry','transport_hub']:
        unit_poi = gpd.sjoin(left_df=unit,right_df=poi.loc[(poi['type']==i),['type','geometry']],predicate='contains').groupby(['id'],as_index=False).size()
        unit = pd.merge(unit,unit_poi,on='id',how='left')
        unit.rename(columns={'size':i[:3]},inplace=True)
        unit.fillna(0,inplace=True)
    unit.to_file(f'D:/urban_hierarchy_congestion/results/hierarchy_identification_results/{city}_loubar_hierarchy_poi.shp')